### Track 3: Agentic Fine-Tuning (AgentTune + AuditKIT)
Ticket triage workflow distillation and GRPO tool-call training.

In [ ]:
!pip install -q "auditkit @ git+https://github.com/Lexsi-Labs/AuditKIT.git"
!pip install -q "git+https://github.com/Lexsi-Labs/AgentTune.git" peft accelerate bitsandbytes datasets trl

import os
if not os.path.exists('lexsi-agentic-triage'):
    !git clone https://github.com/prime3436/lexsi-agentic-triage.git
%cd lexsi-agentic-triage

In [ ]:
from huggingface_hub import login
tok = os.environ.get('HF_TOKEN')
if tok:
    login(token=tok)

In [ ]:
!python generate_dataset.py
!python eval_baseline.py

In [ ]:
import json
from agenttune.agentic import Project, DictToolHarness, ReActStrategy
from agent import TriageBaseline

ag = TriageBaseline()

def assess_hazard(description: str) -> dict:
    r = ag.run(description)
    return {'hazard_detected': r['hazard_detected'], 'hazard_type': r['hazard_type']}

def classify_trade(description: str) -> dict:
    r = ag.run(description)
    return {'category': r['category'], 'severity': r['severity']}

def estimate_cost(category: str, severity: str) -> dict:
    b = {'P1_CRITICAL': (3500, 9500), 'P2_HIGH': (2800, 7500), 'P3_MEDIUM': (800, 3000), 'P4_LOW': (600, 2000)}
    lo, hi = b.get(severity, (800, 3000))
    return {'min_cost_inr': lo, 'max_cost_inr': hi}

def dispatch_contractor(category: str, severity: str) -> dict:
    s = {'P1_CRITICAL': 2, 'P2_HIGH': 6, 'P3_MEDIUM': 24, 'P4_LOW': 72}
    m = {
        'Plumbing': 'Licensed Master Plumber',
        'Electrical': 'Certified Master Electrician',
        'HVAC / Climate Control': 'EPA-Certified HVAC Technician',
        'Safety & Security': 'Licensed General Handyman',
        'General Maintenance': 'Licensed General Handyman',
    }
    return {'specialty': m.get(category, 'Licensed General Handyman'), 'sla_hours': s.get(severity, 24)}

h = DictToolHarness({
    'assess_hazard': assess_hazard,
    'classify_trade': classify_trade,
    'estimate_cost': estimate_cost,
    'dispatch_contractor': dispatch_contractor,
}, max_steps=6)

def pol(s):
    q = getattr(s, 'task', 'sparking outlet in kitchen')
    if s.step == 0:
        return {'name': 'assess_hazard', 'arguments': {'description': q}, 'thought': 'hazard'}
    if s.step == 1:
        return {'name': 'classify_trade', 'arguments': {'description': q}, 'thought': 'trade'}
    if s.step == 2:
        return {'name': 'estimate_cost', 'arguments': {'category': 'Electrical', 'severity': 'P1_CRITICAL'}, 'thought': 'cost'}
    if s.step == 3:
        return {'name': 'dispatch_contractor', 'arguments': {'category': 'Electrical', 'severity': 'P1_CRITICAL'}, 'thought': 'dispatch'}
    return {'name': 'finish', 'arguments': {'answer': 'P1_CRITICAL Electrical'}}

p = Project(strategy=ReActStrategy(pol, max_steps=6), harness=h)
log = p.infer('The socket in the kitchen is sparking and smells like burning plastic')
print(log.tier, len(log.events))
print(p.evaluate_agentic(['The socket in the kitchen is sparking']))
print(len(p.heal()))

In [ ]:
from agenttune import create_distill_trainer

rows = [json.loads(l) for l in open('data/agenttune_train.jsonl', encoding='utf-8')]
demos = [(r['prompt'][-1]['content'], r['answer']) for r in rows]
mdl = 'Qwen/Qwen2.5-1.5B-Instruct'

dt = create_distill_trainer(
    student=mdl,
    demonstrations=demos,
    output_dir='./output/distilled',
    num_train_epochs=3,
    learning_rate=2e-4,
)
print(dt.train())

In [ ]:
from datasets import Dataset
from agenttune.core.backend_factory import create_agentic_trainer

def rew(completions, answer=None, **kw):
    out = []
    for i, c in enumerate(completions):
        txt = str(c)
        g = json.loads(answer[i]) if answer and i < len(answer) else {}
        v = 0.0
        if g.get('category', '') in txt:
            v += 0.35
        if g.get('severity', '') in txt:
            v += 0.35
        if str(g.get('hazard_detected', '')).lower() in txt.lower():
            v += 0.30
        out.append(v)
    return out

ds = Dataset.from_list(rows[:120])

gt = create_agentic_trainer(
    algorithm='grpo',
    model=mdl,
    train_dataset=ds,
    tools=[assess_hazard, classify_trade, estimate_cost, dispatch_contractor],
    reward_funcs=[rew],
    output_dir='./output/grpo',
    max_steps=40,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_steps_per_turn=5,
    use_vllm=False,
    peft_config=dict(r=16, lora_alpha=32, target_modules='all-linear', task_type='CAUSAL_LM'),
    report_to='none',
)
print(gt.train())

In [ ]:
!python run_auditkit_eval.py